<a href="https://colab.research.google.com/github/roeiyanku/UAV_Sound_Classification_Project/blob/main/notebooks/02_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Loading
This section outlines the process of loading and preprocessing audio data for feature extraction. The goal is to prepare the audio for various machine learning models.

## 0) Paths & Constants

In [ ]:
# ── All paths and settings are defined here ──────────────────────────────────

DATASET = "pump"  # "pump" or "valve" — must match what was used in notebook 01
SPLITS_PATH = f"/content/drive/MyDrive/Final Project RMOT/MIMII_dataset/processed/splits/{DATASET}/processed_audio_splits.joblib"

# Set to True to augment training data before extracting features
APPLY_AUGMENTATION = True

if APPLY_AUGMENTATION:
    FEATURES_SAVE_DIR = f"/content/drive/MyDrive/Final Project RMOT/MIMII_dataset/processed/augmented/{DATASET}"
else:
    FEATURES_SAVE_DIR = f"/content/drive/MyDrive/Final Project RMOT/MIMII_dataset/processed/features/{DATASET}"

RANDOM_STATE         = 42
SAMPLE_RATE          = 16000
CLIP_SECONDS         = 5
N_MELS               = 64
TF_BATCH_SIZE        = 32
EMBEDDING_BATCH_SIZE = 4
CNN_EPOCHS           = 10

AUG_TYPES = ['time_stretch', 'pitch_shift', 'add_noise',
             'time_shift',   'gain',         'spec_augment']

### Import Dependencies
This cell imports all necessary Python libraries for audio processing, deep learning models (TensorFlow and PyTorch), and utilities like `joblib` for data loading. Key libraries include `librosa` for audio manipulation, `tensorflow` for dataset creation, and `transformers` for pre-trained AST and Wav2Vec2 models.

In [ ]:
import os
import time
import joblib
import numpy as np
import librosa
import tensorflow as tf
import torch
import numpy as np


from transformers import (
    AutoFeatureExtractor,
    ASTForAudioClassification,
    Wav2Vec2Processor,
    Wav2Vec2Model,
)

### Mount Drive and Load Prepared Data
This section mounts Google Drive to access stored data and then loads pre-processed audio splits (`X_train`, `y_train`, etc.) from a `joblib` file. These splits contain file paths to audio samples and their corresponding labels, ready for feature extraction and model training.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = joblib.load(SPLITS_PATH)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

### Define Constants

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


### Audio Preprocessing Functions
- `load_waveform`: Loads an audio file and pads/trims it to a fixed duration.
- `Data Augmentation` : Defines 6 augmentation types that can be applied to waveforms during feature extraction in the next notebook.


In [ ]:
def load_waveform(path, sr=SAMPLE_RATE, seconds=CLIP_SECONDS):
    """Load one audio file, then pad or trim it to a fixed length."""
    audio, _ = librosa.load(path, sr=sr)
    target_len = sr * seconds

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    return audio.astype(np.float32)

def augment_waveform(y, sr, aug_type):
    """
    Apply one of 6 augmentation strategies to a waveform.

    Parameters
    ----------
    y        : np.ndarray – raw waveform
    sr       : int        – sample rate
    aug_type : str        – one of:
                 'time_stretch', 'pitch_shift', 'add_noise',
                 'time_shift',   'gain',         'spec_augment'

    Returns
    -------
    np.ndarray – augmented waveform (same length as input)
    """
    rng = np.random.default_rng()

    if aug_type == 'time_stretch':
        rate = rng.uniform(0.9, 1.1)
        y = librosa.effects.time_stretch(y, rate=rate)

    elif aug_type == 'pitch_shift':
        n_steps = rng.uniform(-2, 2)
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

    elif aug_type == 'add_noise':
        noise = 0.005 * rng.standard_normal(len(y))
        y = y + noise.astype(y.dtype)

    elif aug_type == 'time_shift':
        shift = int(rng.uniform(0, 0.5) * len(y))
        y = np.roll(y, shift)

    elif aug_type == 'gain':
        factor = rng.uniform(0.7, 1.3)
        y = (y * factor).astype(y.dtype)

    elif aug_type == 'spec_augment':
        S = librosa.stft(y)
        n_freq, n_frames = S.shape
        f0 = rng.integers(0, n_freq // 2)
        S[f0 : f0 + rng.integers(1, max(2, n_freq // 8)), :] = 0
        t0 = rng.integers(0, n_frames // 2)
        S[:, t0 : t0 + rng.integers(1, max(2, n_frames // 8))] = 0
        y = librosa.istft(S, length=len(y))

    else:
        raise ValueError(f"Unknown aug_type: {aug_type!r}")

    return y.astype(np.float32)

In [ ]:
# Quick preview of all 6 types on the first training file

import matplotlib.pyplot as plt
import librosa.display

sample_path = X_train[0]
y_orig, sr_orig = librosa.load(sample_path, sr=None)

aug_types = ['time_stretch', 'pitch_shift', 'add_noise',
             'time_shift',   'gain',         'spec_augment']

fig, axes = plt.subplots(len(aug_types) + 1, 1, figsize=(12, 14))
librosa.display.waveshow(y_orig, sr=sr_orig, ax=axes[0])
axes[0].set_title("Original")

for ax, aug in zip(axes[1:], aug_types):
    y_aug = augment_waveform(y_orig.copy(), sr_orig, aug)
    librosa.display.waveshow(y_aug, sr=sr_orig, ax=ax)
    ax.set_title(aug)

plt.tight_layout()
plt.show()

### Augment Training Data

Build an expanded training set by appending augmented copies of each training file.



In [ ]:
if APPLY_AUGMENTATION:
    print("Augmenting training data...")
    aug_X_train, aug_y_train = [], []

    for path, label in zip(X_train, y_train):
        # keep original
        aug_X_train.append(path)
        aug_y_train.append(label)
        # add one augmented copy per aug type
        for aug in AUG_TYPES:
            aug_X_train.append(path)
            aug_y_train.append(label)

    print(f"Training set: {len(X_train)} -> {len(aug_X_train)} samples")
    X_train_final = aug_X_train
    y_train_final = aug_y_train
else:
    X_train_final = X_train
    y_train_final = y_train

### Acoustic Spectrogram Transformer (AST) Feature Extraction
This section defines and executes the `extract_ast_features` function. This function utilizes a pre-trained AST model from Hugging Face Transformers to extract high-level features from audio waveforms. It processes audio in batches, leveraging the `AutoFeatureExtractor` and `ASTForAudioClassification` models, and computes the mean of the last hidden states as features. The extracted features for training, validation, and test sets are then printed.

In [ ]:
def extract_ast_features(paths, batch_size=EMBEDDING_BATCH_SIZE):
    feature_extractor = AutoFeatureExtractor.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593"
    )
    ast_model = ASTForAudioClassification.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593"
    ).to(device)
    ast_model.eval()

    features = []

    for start_idx in range(0, len(paths), batch_size):
        batch_paths = paths[start_idx:start_idx + batch_size]
        batch_audio = [load_waveform(path) for path in batch_paths]

        inputs = feature_extractor(
            batch_audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = ast_model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]
            batch_features = hidden_states.mean(dim=1).detach().cpu().numpy()

        features.append(batch_features)

    return np.vstack(features)


ast_start = time.time()
ast_train_features = extract_ast_features(X_train)
ast_val_features = extract_ast_features(X_val)
ast_test_features = extract_ast_features(X_test)

ast_train_time = time.time() - ast_start

print("AST feature shapes:", ast_train_features.shape, ast_val_features.shape, ast_test_features.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

### Wav2Vec2 Feature Extraction
Similar to the AST section, this block defines and executes the `extract_wav2vec_features` function. This function uses a pre-trained Wav2Vec2 model to extract audio features from raw waveforms. It uses `Wav2Vec2Processor` to prepare the audio and `Wav2Vec2Model` to obtain hidden states, which are then averaged to form the final features. The extracted features for training, validation, and test sets are printed along with the time taken for extraction.

In [ ]:
def extract_wav2vec_features(paths, batch_size=EMBEDDING_BATCH_SIZE):
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
    wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)
    wav2vec_model.eval()

    features = []

    for start_idx in range(0, len(paths), batch_size):
        batch_paths = paths[start_idx:start_idx + batch_size]
        batch_audio = [load_waveform(path) for path in batch_paths]

        inputs = processor(
            batch_audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=True,
        )

        input_values = inputs.input_values.to(device)
        attention_mask = (
            inputs.attention_mask.to(device)
            if hasattr(inputs, "attention_mask")
            else None
        )

        with torch.no_grad():
            outputs = wav2vec_model(
                input_values=input_values,
                attention_mask=attention_mask,
            )
            hidden_states = outputs.last_hidden_state
            batch_features = hidden_states.mean(dim=1).detach().cpu().numpy()

        features.append(batch_features)

    return np.vstack(features)


wav2vec_start = time.time()
wav2vec_train_features = extract_wav2vec_features(X_train)
wav2vec_val_features = extract_wav2vec_features(X_val)
wav2vec_test_features = extract_wav2vec_features(X_test)

wav2vec_train_time = time.time() - wav2vec_start

print("Wav2Vec2 feature shapes:", wav2vec_train_features.shape, wav2vec_val_features.shape, wav2vec_test_features.shape)


### logmel Feature Extraction


In [ ]:
def extract_logmel_features(paths, sample_rate=SAMPLE_RATE, n_mels=64, duration=2.0):
    features = []
    target_len = int(sample_rate * duration)

    for path in paths:
        y, sr = librosa.load(path, sr=sample_rate)

        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=n_mels
        )

        logmel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        features.append(logmel)

    return np.stack(features).astype(np.float32)

logmel_start = time.time()
logmel_train_features = extract_logmel_features(X_train)
logmel_val_features = extract_logmel_features(X_val)
logmel_test_features = extract_logmel_features(X_test)

logmel_train_time = time.time() - logmel_start

print(
    "Log-Mel feature shapes:",
    logmel_train_features.shape,
    logmel_val_features.shape,
    logmel_test_features.shape
)

### Save Extracted Features
This section saves the extracted AST and Wav2Vec2 features, into *two separate `joblib` files*. The labels are also saved into a `joblib` file. This allows for persistent storage of these processed features, enabling their reuse in subsequent machine learning model training without re-running the feature extraction process.

In [ ]:
ast_feature_data = {
    "ast_train_features": ast_train_features,
    "ast_val_features":   ast_val_features,
    "ast_test_features":  ast_test_features
}

joblib.dump(ast_feature_data, os.path.join(FEATURES_SAVE_DIR, "extracted_ast_features.joblib"))
print("Extracted AST features saved to:", FEATURES_SAVE_DIR)

In [ ]:
wav2vec_feature_data = {
    "wav2vec_train_features": wav2vec_train_features,
    "wav2vec_val_features":   wav2vec_val_features,
    "wav2vec_test_features":  wav2vec_test_features,
}

joblib.dump(wav2vec_feature_data, os.path.join(FEATURES_SAVE_DIR, "extracted_wav2vec_features.joblib"))
print("Extracted Wav2Vec2 features saved to:", FEATURES_SAVE_DIR)

In [ ]:
logmel_features_data = {
    "logmel_train_features": logmel_train_features,
    "logmel_val_features":   logmel_val_features,
    "logmel_test_features":  logmel_test_features,
}

joblib.dump(logmel_features_data, os.path.join(FEATURES_SAVE_DIR, "extracted_logmel_features.joblib"))
print("Extracted logmel features saved to:", FEATURES_SAVE_DIR)

In [ ]:
labels = {
    "train": y_train_final,
    "val":   y_val,
    "test":  y_test
}

joblib.dump(labels, os.path.join(FEATURES_SAVE_DIR, "labels.joblib"))
print("Labels saved to:", FEATURES_SAVE_DIR)

In [7]:
import nbformat

path = "/content/drive/MyDrive/Final Project RMOT/notebooks/02_feature_extraction.ipynb"

nb = nbformat.read(path, as_version=4)

# Remove notebook-level widget metadata
nb.metadata.pop("widgets", None)

# Clean cell outputs and any cell-level widget metadata
for cell in nb.cells:
    if cell.cell_type == "code":
        cell["outputs"] = []
        cell["execution_count"] = None
        # Some Colab notebooks also stash widget state in cell metadata
        cell.metadata.pop("widgets", None)

nbformat.write(nb, path)
print("Notebook cleaned")

Notebook cleaned


In [8]:
import json
with open(path) as f:
    nb_raw = json.load(f)
print("widgets in metadata?", "widgets" in nb_raw.get("metadata", {}))

widgets in metadata? False
